# Assignment Beta - Question 2 (Student ID: 23I-0124)
## Fine-tuning a small LLM for e-commerce tool calling with LoRA, QLoRA, DoRA and AdaLoRA

**How to run:** Runtime > Change runtime type > **T4 GPU**. Then run the cells from top to bottom (Shift+Enter).
Results are saved in your Google Drive folder `Assignment_Q2`, so if Colab disconnects you can re-run cells 2-4 and the finished runs are skipped.

**Part A (Section 2)** is in this notebook. Section 3 cells are added later.

### Cell 2 - Install libraries (if Colab asks to *Restart session*, click it, then continue from Cell 3)

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets

### Cell 3 - Settings (same for EVERY run, only the adapter method changes)

In [ ]:
import os, re, json, time, gc
import numpy as np, pandas as pd, torch
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, AdaLoraConfig, get_peft_model, prepare_model_for_kbit_training

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Assignment_Q2'
os.makedirs(OUT, exist_ok=True)
RES = f"{OUT}/results.csv"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42
N_TOTAL, N_TEST = 2000, 200          # 1800 train + 200 test
MAX_LEN = 640                        # skip very long examples so the JSON answer is never cut
EPOCHS = 1
MICRO_BS, GRAD_ACC = 4, 4            # effective batch size = 16
LR = 2e-4
WARMUP = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
DTYPE = torch.float16                # T4 does not support bf16, so we use fp16
AMP = (DTYPE == torch.float16)
GEN_BS, MAX_NEW = 16, 160

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

### Cell 4 - Load 2,000 examples from glaive-function-calling-v2 and split 1,800 train / 200 test (test set is never used in training)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

INSTR = ("\n\nWhen a function is needed, reply with ONLY a JSON object of the form "
         '{"name": "<function name>", "arguments": {<arguments>}} and nothing else. Do not add any explanation.')

def prompt_text(ex):
    msgs = [{"role": "system", "content": ex["system"]}, {"role": "user", "content": ex["user"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_example(ex):
    """Take the FIRST user message and the FIRST assistant function call from a glaive chat."""
    system, chat = ex["system"], ex["chat"]
    m = re.search(r"USER:\s*((?:(?!ASSISTANT:).)*?)\s*ASSISTANT:\s*<functioncall>\s*(.*?)\s*<\|endoftext\|>", chat, re.S)
    if not m:
        return None
    user, call = m.group(1), m.group(2)
    nm = re.search(r'"name":\s*"([^"]+)"', call)
    am = re.search(r'"arguments":\s*(.*)\}\s*$', call, re.S)
    if not nm or not am:
        return None
    a = am.group(1).strip()
    if a.startswith("'") and a.endswith("'"):
        a = a[1:-1]
    try:
        args = json.loads(a)
    except Exception:
        return None
    if not isinstance(args, dict):
        return None
    allowed = re.findall(r'\{\s*"name":\s*"([^"]+)",\s*"description"', system)
    if nm.group(1) not in allowed:
        return None
    sys_text = system.replace("SYSTEM:", "", 1).strip() + INSTR
    return {"system": sys_text, "user": user, "name": nm.group(1), "args": args, "allowed": allowed}

SPLIT_FILE = f"{OUT}/data_split.json"
if os.path.exists(SPLIT_FILE):
    split = json.load(open(SPLIT_FILE))
    print("Loaded existing split from Drive")
else:
    from datasets import load_dataset
    ds = load_dataset("glaiveai/glaive-function-calling-v2", split="train").shuffle(seed=SEED)
    good = []
    for ex in ds:
        p = parse_example(ex)
        if p is None:
            continue
        p["target"] = json.dumps({"name": p["name"], "arguments": p["args"]}, ensure_ascii=False)
        n = (len(tok(prompt_text(p), add_special_tokens=False)["input_ids"])
             + len(tok(p["target"] + tok.eos_token, add_special_tokens=False)["input_ids"]))
        if n > MAX_LEN:
            continue
        good.append(p)
        if len(good) == N_TOTAL:
            break
    split = {"train": good[:N_TOTAL - N_TEST], "test": good[N_TOTAL - N_TEST:]}
    json.dump(split, open(SPLIT_FILE, "w"))

train_ex, test_ex = split["train"], split["test"]
print("train:", len(train_ex), "test:", len(test_ex))
print("Example user message :", test_ex[0]["user"][:150])
print("Example target answer:", test_ex[0]["target"][:200])

### Cell 5 - Helper functions (training loop, evaluation, scoring). Just run it.

In [ ]:
def encode_train(ex):
    p = tok(prompt_text(ex), add_special_tokens=False)["input_ids"]
    t = tok(ex["target"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + t, "labels": [-100] * len(p) + t}   # loss only on the JSON answer

def collate(items):
    L = max(len(i["input_ids"]) for i in items)
    ids = torch.full((len(items), L), tok.pad_token_id)
    lab = torch.full((len(items), L), -100)
    att = torch.zeros((len(items), L), dtype=torch.long)
    for k, i in enumerate(items):
        n = len(i["input_ids"])
        ids[k, :n] = torch.tensor(i["input_ids"]); lab[k, :n] = torch.tensor(i["labels"]); att[k, :n] = 1
    return {"input_ids": ids.cuda(), "labels": lab.cuda(), "attention_mask": att.cuda()}

def load_base(quant=False):
    kw = dict(dtype=DTYPE, device_map={"": 0})
    if quant:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw)

def make_peft(model, method, r, total_steps):
    if method == "adalora":
        cfg = AdaLoraConfig(init_r=int(1.5 * r), target_r=r, lora_alpha=2 * r, lora_dropout=0.05,
                            target_modules=TARGET_MODULES, task_type="CAUSAL_LM",
                            tinit=int(0.1 * total_steps), tfinal=int(0.3 * total_steps),
                            deltaT=max(1, int(0.03 * total_steps)), total_step=total_steps)
    else:
        cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules=TARGET_MODULES,
                         task_type="CAUSAL_LM", use_dora=(method == "dora"))
    return get_peft_model(model, cfg)

def train(model, method, total_steps):
    enc = [encode_train(e) for e in train_ex]
    params = [p for p in model.parameters() if p.requires_grad]
    for p in params:
        if p.dtype != torch.float32:
            p.data = p.data.float()
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    warm = max(1, int(WARMUP * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total_steps - s) / (total_steps - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    g = torch.Generator().manual_seed(SEED)
    per_step = MICRO_BS * GRAD_ACC
    model.train()
    step, losses = 0, []
    for ep in range(EPOCHS):
        perm = torch.randperm(len(enc), generator=g).tolist()
        for s in range(0, (len(perm) // per_step) * per_step, per_step):
            chunk = perm[s:s + per_step]
            tot = 0.0
            for m in range(0, len(chunk), MICRO_BS):
                batch = collate([enc[i] for i in chunk[m:m + MICRO_BS]])
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    out = model(**batch, use_cache=False)
                loss = out.loss / GRAD_ACC
                scaler.scale(loss).backward()
                tot += loss.item()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            if method == "adalora" and torch.isfinite(gn):
                model.base_model.update_and_allocate(step)
            opt.zero_grad(set_to_none=True)
            step += 1; losses.append(tot)
            if step % 10 == 0 or step == total_steps:
                print(f"step {step}/{total_steps}  loss {np.mean(losses[-10:]):.4f}")
    return losses

def score_output(ex, out):
    r = dict(valid=False, name_ok=False, full_ok=False, made_up=False, pred_name=None, pred_args=None)
    try:
        obj = json.loads(out.strip())
    except Exception:
        return r
    r["valid"] = True
    if not isinstance(obj, dict):
        return r
    r["pred_name"] = obj.get("name")
    args = obj.get("arguments")
    if isinstance(args, str):
        try: args = json.loads(args)
        except Exception: pass
    r["pred_args"] = args
    r["name_ok"] = (r["pred_name"] == ex["name"])
    r["made_up"] = (r["pred_name"] not in ex["allowed"])
    r["full_ok"] = r["name_ok"] and (args == ex["args"])
    return r

@torch.no_grad()
def evaluate(model, run_name):
    model.eval()
    order = sorted(range(len(test_ex)), key=lambda i: len(test_ex[i]["system"]) + len(test_ex[i]["user"]))
    outs = {}
    torch.cuda.synchronize(); t0 = time.time()
    for s in range(0, len(order), GEN_BS):
        idx = order[s:s + GEN_BS]
        b = tok([prompt_text(test_ex[i]) for i in idx], return_tensors="pt", padding=True,
                add_special_tokens=False).to("cuda")
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            g = model.generate(**b, max_new_tokens=MAX_NEW, do_sample=False,
                               pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for i, row in zip(idx, g[:, b["input_ids"].shape[1]:]):
            outs[i] = tok.decode(row, skip_special_tokens=True)
    torch.cuda.synchronize(); infer_sec = time.time() - t0
    preds = []
    for i in range(len(test_ex)):
        sc = score_output(test_ex[i], outs[i])
        preds.append(dict(idx=i, user=test_ex[i]["user"], gold_name=test_ex[i]["name"],
                          gold_args=test_ex[i]["args"], output=outs[i], **sc))
    json.dump(preds, open(f"{OUT}/predictions_{run_name}.json", "w"), indent=1, default=str)
    print("First 2 raw outputs:", [outs[0][:150], outs[1][:150]])
    return preds, infer_sec

def log_result(row):
    new = pd.DataFrame([row])
    if os.path.exists(RES):
        df = pd.read_csv(RES)
        df = pd.concat([df[df["run"] != row["run"]], new], ignore_index=True)
    else:
        df = new
    df.to_csv(RES, index=False)

def summarize(run_name, method, rank, trainable, total, peak_train, train_min, preds, infer_sec, peak_infer):
    n = len(preds)
    row = dict(run=run_name, method=method, rank=rank, trainable_params=trainable, total_params=total,
               trainable_pct=100 * trainable / total, peak_mem_train_gb=peak_train, train_minutes=train_min,
               json_valid=sum(p["valid"] for p in preds), tool_correct=sum(p["name_ok"] for p in preds),
               full_match=sum(p["full_ok"] for p in preds), made_up_names=sum(p["made_up"] for p in preds),
               infer_sec_200=infer_sec, peak_mem_infer_gb=peak_infer)
    row["json_valid_pct"] = 100 * row["json_valid"] / n
    row["tool_acc_pct"] = 100 * row["tool_correct"] / n
    row["full_match_pct"] = 100 * row["full_match"] / n
    log_result(row)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in row.items()})

def already_done(run_name):
    return os.path.exists(RES) and run_name in set(pd.read_csv(RES)["run"])

def free():
    gc.collect(); torch.cuda.empty_cache()

def run_base():
    if already_done("base"):
        print("base already done - skipping"); return
    set_seed(SEED); free(); torch.cuda.reset_peak_memory_stats()
    model = load_base()
    total = sum(p.numel() for p in model.parameters())
    preds, infer_sec = evaluate(model, "base")
    summarize("base", "none (no fine-tuning)", 0, 0, total, float("nan"), 0.0, preds, infer_sec,
              torch.cuda.max_memory_allocated() / 2**30)
    del model; free()

def run_experiment(run_name, method, r):
    if already_done(run_name):
        print(run_name, "already done - skipping"); return
    set_seed(SEED); free(); torch.cuda.reset_peak_memory_stats()
    total_steps = (len(train_ex) // (MICRO_BS * GRAD_ACC)) * EPOCHS
    gc_kw = {"use_reentrant": False}
    base = load_base(quant=(method == "qlora"))
    if method == "qlora":
        base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                                               gradient_checkpointing_kwargs=gc_kw)
    else:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gc_kw)
        base.enable_input_require_grads()
    model = make_peft(base, method, r, total_steps)
    trainable, total = model.get_nb_trainable_parameters()
    print(f"{run_name}: trainable {trainable:,} of {total:,} ({100 * trainable / total:.3f}%)")
    t0 = time.time()
    train(model, method, total_steps)
    train_min = (time.time() - t0) / 60
    peak_train = torch.cuda.max_memory_allocated() / 2**30
    torch.cuda.reset_peak_memory_stats()
    preds, infer_sec = evaluate(model, run_name)
    peak_infer = torch.cuda.max_memory_allocated() / 2**30
    model.save_pretrained(f"{OUT}/adapters/{run_name}")
    if method == "adalora":
        try:
            rp = model.peft_config["default"].rank_pattern
            json.dump({k: int(sum(v)) for k, v in rp.items()}, open(f"{OUT}/adalora_rank_pattern_{run_name}.json", "w"), indent=1)
        except Exception as e:
            print("could not save rank pattern:", e)
    summarize(run_name, method, r, trainable, total, peak_train, train_min, preds, infer_sec, peak_infer)
    del model, base; free()

### Cell 6 - Run 0: the ORIGINAL model with no fine-tuning (the 'before' picture). About 3-5 minutes.

In [ ]:
run_base()

### Cell 7 - Run 1: LoRA (bf16/fp16, r=8). About 10-15 minutes. Keep the browser tab open.

In [ ]:
run_experiment("lora_r8", "lora", 8)

### Cell 8 - Run 2: QLoRA (4-bit NF4, r=8)

In [ ]:
run_experiment("qlora_r8", "qlora", 8)

### Cell 9 - Run 3: DoRA (r=8). Slower than LoRA, this is normal.

In [ ]:
run_experiment("dora_r8", "dora", 8)

### Cell 10 - Run 4: AdaLoRA (start r=12, target r=8)

In [ ]:
run_experiment("adalora_r8", "adalora", 8)

### Cell 11 - Final comparison table (copy or screenshot this and send it to Claude)

In [ ]:
df = pd.read_csv(RES)
order = ["base", "lora_r8", "qlora_r8", "dora_r8", "adalora_r8"]
df = df[df["run"].isin(order)].set_index("run").loc[[o for o in order if o in set(df["run"])]].reset_index()
table = pd.DataFrame({
    "Run": df["run"], "Method": df["method"], "Rank": df["rank"],
    "Trainable params": df["trainable_params"].map("{:,}".format),
    "Trainable %": df["trainable_pct"].round(3),
    "Peak GPU mem (GB)": df["peak_mem_train_gb"].round(2),
    "Train time (min)": df["train_minutes"].round(1),
    "JSON valid /200": df["json_valid"],
    "Tool acc %": df["tool_acc_pct"].round(1),
    "Full match %": df["full_match_pct"].round(1),
    "Made-up fn names": df["made_up_names"],
    "Inference 200 ex (s)": df["infer_sec_200"].round(1),
})
table